# Camada Gold — SISCAN / 10ª CRS

Processamento analítico de 2020 a 2025. A dimensão populacional do Censo 2022 depende do notebook 06.

As seções 1–6 preparam, validam e gravam os dez fatos/dimensões deste notebook. As seções 7–11 são análises exploratórias somente leitura. Fatos externos de Outubro Rosa são preservados. Nulos são mantidos, exceto preenchimentos explicitamente justificados pelo TABNET.

In [0]:
from collections.abc import Sequence
from math import sqrt

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
ANO_INICIO, ANO_FIM = 2020, 2025
VALIDAR_REGRESSAO = True
AUDITAR_OUTUBRO_ROSA_LEGADO = False
GRUPO_ALVO = "25 a 64 anos (População-Alvo)"
GRUPO_SOBRE = "<25 anos (Sobrerrastreamento)"
GRUPO_SUB = "65 anos ou mais (Sub-rastreamento)"
FAIXAS_25_64 = tuple(f"{age} a {age + 4} anos" for age in range(25, 61, 5))
FAIXAS_MENOR_25 = ("Até 9 anos", "10 a 14 anos", "15 a 19 anos", "20 a 24 anos")
FAIXAS_65_MAIS = ("65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos")
ALTO_GRAU = ["alto_grau_HPV-NIC I", "micro-invasa", "carcinoma-epiderm"]
META_ANUAL_PCT = 33.0

fatos_preparados: dict[str, DataFrame] = {}
contagens_preparadas: dict[str, int] = {}

def nome_tabela(schema: str, tabela: str) -> str:
    return f"{CATALOG}.{schema}.{tabela}"

def ler_silver(tabela: str) -> DataFrame:
    return spark.table(nome_tabela(SCHEMA_SILVER, tabela))

def ler_gold(tabela: str) -> DataFrame:
    if tabela in fatos_preparados:
        return fatos_preparados[tabela]
    return spark.table(nome_tabela(SCHEMA_GOLD, tabela))

def percentual(numerador: Column, denominador: Column) -> Column:
    return F.when(denominador > 0, F.round((numerador / denominador) * 100, 2))

def codigo_ibge() -> Column:
    return F.lpad(F.col("codigo_municipio").cast("string"), 6, "0")

def grupo_etario_3() -> Column:
    faixa = F.col("faixa_etaria")
    return (
        F.when(faixa.isin(*FAIXAS_MENOR_25), F.lit("<= 24 anos"))
        .when(faixa.isin(*FAIXAS_25_64), F.lit("25 a 64 anos"))
        .when(faixa.isin(*FAIXAS_65_MAIS), F.lit(">= 65 anos"))
        .otherwise(F.lit("Ignorado"))
    )

def exigir(condicao: bool, mensagem: str) -> None:
    if not condicao:
        raise ValueError(mensagem)

def validar_regressao(df: DataFrame, tabela: str) -> None:
    nome = nome_tabela(SCHEMA_GOLD, tabela)
    if not VALIDAR_REGRESSAO or not spark.catalog.tableExists(nome):
        return

    tecnicas = {"_silver_source", "_gold_source", "_gold_processed_time"}
    atual = spark.table(nome)
    colunas = [name for name in df.columns if name not in tecnicas]
    atuais = set(atual.columns) - tecnicas
    exigir(
        atuais == set(colunas),
        f"{tabela}: colunas ausentes={sorted(atuais - set(colunas))}, "
        f"novas={sorted(set(colunas) - atuais)}",
    )
    propostas = df.select(*colunas)
    publicadas = atual.select(*colunas)
    tipos_atuais = {field.name: field.dataType for field in publicadas.schema.fields}
    mudancas_tipo = [
        field.name for field in propostas.schema.fields
        if field.dataType != tipos_atuais[field.name]
    ]
    exigir(not mudancas_tipo, f"{tabela}: tipos alterados em {mudancas_tipo}")
    removidas = publicadas.exceptAll(propostas).count()
    novas = propostas.exceptAll(publicadas).count()
    exigir(
        not removidas and not novas,
        f"{tabela}: regressão — {removidas} registros removidos/alterados; "
        f"{novas} registros novos/alterados",
    )

def validar_e_preparar(
    df: DataFrame,
    tabela_gold: str,
    pk_cols: Sequence[str],
    obrigatorio_cols: Sequence[str],
) -> int:
    obrigatorias = tuple(dict.fromkeys([*pk_cols, *obrigatorio_cols]))
    metricas = [F.count("*").alias("linhas")]
    for name in obrigatorias:
        col = F.col(name)
        metricas.append(
            F.count(F.when(col.isNull() | (F.trim(col.cast("string")) == ""), 1)).alias(name)
        )
    resumo = df.agg(*metricas).first().asDict()
    exigir(resumo["linhas"] > 0, f"{tabela_gold}: tabela vazia")
    invalidas = {name: resumo[name] for name in obrigatorias if resumo[name]}
    exigir(not invalidas, f"{tabela_gold}: campos obrigatórios inválidos: {invalidas}")
    dups = df.groupBy(*pk_cols).count().filter(F.col("count") > 1).count()
    exigir(dups == 0, f"{tabela_gold}: {dups} chaves duplicadas em {list(pk_cols)}")
    validar_regressao(df, tabela_gold)
    fatos_preparados[tabela_gold] = df
    contagens_preparadas[tabela_gold] = resumo["linhas"]
    print(f"{tabela_gold}: {resumo['linhas']} linhas validadas.")
    return resumo["linhas"]

def gravar_fatos() -> None:
    for tabela, df in fatos_preparados.items():
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(nome_tabela(SCHEMA_GOLD, tabela))
        )
        print(f"{tabela}: {contagens_preparadas[tabela]} linhas gravadas.")

def wilson(k: float, n: float, z: float = 1.96) -> tuple[float, float]:
    """Intervalo de Wilson em percentual."""
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")

dependencias = [
    "silver.exames_geral", "silver.faixa_etaria", "silver.adequabilidade",
    "silver.resultados_citopatologicos", "silver.resultados_categoria_anual",
    "silver.resultados_categoria_mensal", "silver.resultados_categoria_faixa_etaria",
    "gold.dim_populacao_municipios",
]
ausentes = [
    nome for nome in dependencias
    if not spark.catalog.tableExists(f"{CATALOG}.{nome}")
]
exigir(not ausentes, f"Dependências não disponíveis: {ausentes}")

## 1. Série temporal e territorial

`fato_exames_temporais_territoriais` — município × competência. Participação = exames do município / total regional × 100. Variações absoluta e percentual comparam competências consecutivas. Agosto/2022 ausente; setembro/2022 não possui referência mensal anterior.

In [0]:
df_silver_exames = ler_silver("exames_geral")

win_regional = Window.partitionBy("dt_competencia")
win_lag_muni = Window.partitionBy("codigo_municipio").orderBy("dt_competencia")

df_q1_q2 = (
    df_silver_exames
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .withColumn("total_exames_10crs", F.sum("quantidade").over(win_regional))
    .withColumn(
        "participacao_territorial_pct",
        percentual(F.col("quantidade"), F.col("total_exames_10crs")),
    )
    # O lag só é válido para competências consecutivas.
    .withColumn("_dt_lag", F.lag("dt_competencia", 1).over(win_lag_muni))
    .withColumn("_qtde_lag", F.lag("quantidade", 1).over(win_lag_muni))
    .withColumn(
        "quantidade_mes_anterior",
        F.when(F.add_months(F.col("_dt_lag"), 1) == F.col("dt_competencia"), F.col("_qtde_lag")),
    )
    .withColumn("variacao_absoluta", F.col("quantidade") - F.col("quantidade_mes_anterior"))
    .withColumn(
        "variacao_percentual_pct",
        F.when(F.col("quantidade_mes_anterior") > 0,
               F.round(((F.col("quantidade") - F.col("quantidade_mes_anterior"))
                        / F.col("quantidade_mes_anterior")) * 100, 2)),
    )
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "mes", "dt_competencia",
        "quantidade", "total_exames_10crs", "participacao_territorial_pct",
        "quantidade_mes_anterior", "variacao_absoluta", "variacao_percentual_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_preparar(
    df_q1_q2,
    "fato_exames_temporais_territoriais",
    pk_cols=["codigo_municipio", "dt_competencia"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "mes", "dt_competencia"],
)

display(df_q1_q2.orderBy("codigo_municipio", "dt_competencia").limit(12))

## 2. Produção no recorte Outubro Rosa

`fato_outubro_rosa_producao_municipio_ano` — município × ano. O total set–dez exige quatro competências; a variação compara outubro com setembro. São exames registrados no período, não atendimentos identificados como campanha nem evidência causal.

In [0]:
fonte_or = ler_gold("fato_exames_temporais_territoriais")
fonte_or = fonte_or.filter(
    F.col("ano").between(ANO_INICIO, ANO_FIM) & F.col("mes").isin(9, 10, 11, 12)
)
meses_or = (fonte_or.groupBy("codigo_municipio", "municipio", "ano", "mes")
    .agg(F.sum("quantidade").alias("quantidade_mes")))

# Competências ausentes continuam nulas; o total exige os quatro meses.
producao_or = (meses_or.groupBy("codigo_municipio", "municipio", "ano")
    .pivot("mes", [9, 10, 11, 12])
    .agg(F.first("quantidade_mes")))
for n_mes, nome_mes in [(9, "setembro"), (10, "outubro"), (11, "novembro"), (12, "dezembro")]:
    producao_or = producao_or.withColumnRenamed(str(n_mes), f"exames_{nome_mes}")
col_meses = [f"exames_{nome}" for nome in ("setembro", "outubro", "novembro", "dezembro")]
producao_or = (producao_or
    .withColumn("meses_observados", sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in col_meses))
    .withColumn(
        "exames_set_dez",
        F.when(F.col("meses_observados") == 4, sum(F.col(c) for c in col_meses))
    )
    .withColumn(
        "variacao_set_out_pct",
        F.when(F.col("exames_setembro") > 0,
            F.round(100 * (F.col("exames_outubro") - F.col("exames_setembro"))
                    / F.col("exames_setembro"), 2))
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_gold_source", F.lit("gold.fato_exames_temporais_territoriais"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", *col_meses,
        "meses_observados", "exames_set_dez", "variacao_set_out_pct",
        "_gold_source", "_gold_processed_time"
    )
)
validar_e_preparar(
    producao_or,
    "fato_outubro_rosa_producao_municipio_ano",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "meses_observados"],
)
print("Competências set–dez incompletas (total anual mantido nulo):",
      producao_or.filter(F.col("meses_observados") < 4).count())
display(producao_or.orderBy("municipio", "ano"))

### 2b. Perfil etário consolidado — auditoria opcional

`fato_outubro_rosa_grupo_etario` (33 linhas, 56.425 exames na validação histórica) foi materializada externamente a partir de uma extração independente. Não é produzida pelo notebook Silver atual. Auditoria depende de `silver.outubro_rosa_faixa_municipio`; o consolidado não possui dimensão temporal.

In [0]:
dependencias_legado = ["gold.fato_outubro_rosa_grupo_etario","silver.outubro_rosa_faixa_municipio"]
if AUDITAR_OUTUBRO_ROSA_LEGADO:
    ausentes = [nome for nome in dependencias_legado if not spark.catalog.tableExists(f"{CATALOG}.{nome}")]
    if ausentes:
        print(f"Auditoria histórica não executada: {ausentes}")
    else:
        df_or_ge = ler_gold("fato_outubro_rosa_grupo_etario")

        resumo_ge = df_or_ge.agg(
            F.count("*").alias("linhas"),
            F.countDistinct("municipio").alias("municipios"),
            F.countDistinct("grupo_etario").alias("grupos"),
            F.countDistinct("municipio", "grupo_etario").alias("pares_unicos"),
            F.sum("quantidade_exames").alias("total_exames"),
        ).collect()[0]
        assert resumo_ge["linhas"] == 33, \
            f"Esperadas 33 linhas (11 municípios × 3 grupos); obtidas {resumo_ge['linhas']}"
        assert resumo_ge["linhas"] == resumo_ge["pares_unicos"], "Duplicados na PK município × grupo"
        assert resumo_ge["municipios"] == 11 and resumo_ge["grupos"] == 3, "Grade 11 × 3 incompleta"
        assert resumo_ge["total_exames"] == 56425, \
            "Total deve fechar em 56.425 (56.426 do arquivo − 1 'Ignorado' em Quaraí)"
        print(f"Gold grupo_etario: {resumo_ge['linhas']} linhas | "
              f"{resumo_ge['municipios']} municípios × {resumo_ge['grupos']} grupos | "
              f"total {resumo_ge['total_exames']:,} exames")

        FAIXAS_25_64_Q02 = [f"{i} a {i+4} anos" for i in range(25, 61, 5)]
        silver_ge_recalc = (
            ler_silver("outubro_rosa_faixa_municipio")
            .withColumn(
                "grupo_etario",
                F.when(F.col("faixa_etaria").isin("10 a 14 anos", "15 a 19 anos", "20 a 24 anos"), "<25 anos")
                 .when(F.col("faixa_etaria").isin(*FAIXAS_25_64_Q02), "25-64 anos")
                 .when(F.col("faixa_etaria").isin("65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos"), ">=65 anos")
                 .otherwise(None),  # 'Ignorado' — excluído por design (não vira grupo)
            )
            .filter(F.col("grupo_etario").isNotNull())
            .groupBy("municipio", "grupo_etario")
            .agg(F.sum("quantidade").alias("qtd_silver"))
        )
        diverg_ge = (
            silver_ge_recalc
            .join(df_or_ge, ["municipio", "grupo_etario"], "inner")
            .filter(F.col("qtd_silver") != F.col("quantidade_exames"))
            .count()
        )
        assert diverg_ge == 0, f"Cross-check Silver × Gold: {diverg_ge} pares divergentes"
        print("Cross-check Silver × Gold: 33/33 pares conferem valor a valor; "
              "'Ignorado' (1 caso em Quaraí) excluído por design.")

        display(df_or_ge.orderBy("municipio", "grupo_etario"))
else:
    print("Auditoria histórica desabilitada; dados publicados preservados.")

### 2c. Estimativa IPF — auditoria opcional

`fato_outubro_rosa_estimativa_ipf_municipio_mes` é estimativa, não dado observado (341 linhas, 56.425 exames na validação histórica). Estimativa = soma de N(município, faixa) × N(competência, faixa) / N(faixa), assumindo independência condicional entre município e competência dada a faixa. Arredondamento por maiores restos conserva as margens municipais.

A auditoria depende das margens históricas `silver.outubro_rosa_faixa_municipio` e `silver.outubro_rosa_mes_faixa`. A tabela `fato_outubro_rosa_mensal_grupo_etario`, consumida pelo Streamlit, também foi materializada externamente. Esses fatos são preservados; não substituí-los por aproximações com recorte diferente.

In [0]:
dependencias_legado = ["gold.fato_outubro_rosa_estimativa_ipf_municipio_mes","silver.outubro_rosa_faixa_municipio","silver.outubro_rosa_mes_faixa"]
if AUDITAR_OUTUBRO_ROSA_LEGADO:
    ausentes = [nome for nome in dependencias_legado if not spark.catalog.tableExists(f"{CATALOG}.{nome}")]
    if ausentes:
        print(f"Auditoria histórica não executada: {ausentes}")
    else:
        df_or_ipf = ler_gold("fato_outubro_rosa_estimativa_ipf_municipio_mes")

        resumo_ipf = df_or_ipf.agg(
            F.count("*").alias("linhas"),
            F.countDistinct("codigo_municipio").alias("municipios"),
            F.countDistinct("dt_competencia").alias("competencias"),
            F.countDistinct("codigo_municipio", "dt_competencia").alias("pares_unicos"),
            F.sum("quantidade_exames_estimada").alias("total_estimado"),
            F.min(F.col("fl_estimativa").cast("int")).alias("flag_estimativa_min"),
        ).collect()[0]
        assert resumo_ipf["linhas"] == 341, \
            f"Esperadas 341 linhas (11 × 31); obtidas {resumo_ipf['linhas']}"
        assert resumo_ipf["linhas"] == resumo_ipf["pares_unicos"], "Duplicados na PK município × competência"
        assert resumo_ipf["municipios"] == 11 and resumo_ipf["competencias"] == 31, "Grade 11 × 31 incompleta"
        assert resumo_ipf["total_estimado"] == 56425, "Total deve fechar em 56.425 (margens − 1 'Ignorado')"
        assert resumo_ipf["flag_estimativa_min"] == 1, "fl_estimativa deve ser TRUE em todas as linhas"
        print(f"Gold IPF: {resumo_ipf['linhas']} linhas | {resumo_ipf['municipios']} municípios × "
              f"{resumo_ipf['competencias']} competências | total {resumo_ipf['total_estimado']:,} exames")

        silver_ipf_mun = (
            ler_silver("outubro_rosa_faixa_municipio")
            .filter(F.col("faixa_etaria") != "Ignorado")
            .groupBy("codigo_municipio")
            .agg(F.sum("quantidade").alias("total_silver"))
        )
        diverg_ipf_mun = (
            df_or_ipf.groupBy("codigo_municipio")
            .agg(F.sum("quantidade_exames_estimada").alias("total_estimado"))
            .join(silver_ipf_mun, "codigo_municipio")
            .filter(F.col("total_estimado") != F.col("total_silver"))
            .count()
        )
        assert diverg_ipf_mun == 0, f"Totais municipais divergentes da margem Silver: {diverg_ipf_mun}"
        print("Totais municipais: 11/11 idênticos à margem Silver (arredondamento por maiores restos).")

        silver_ipf_comp = (
            ler_silver("outubro_rosa_mes_faixa")
            .filter(F.col("faixa_etaria") != "Ignorado")
            .groupBy("dt_competencia")
            .agg(F.sum("quantidade").alias("total_silver"))
        )
        desvio_ipf_comp = (
            df_or_ipf.groupBy("dt_competencia")
            .agg(F.sum("quantidade_exames_estimada").alias("total_estimado"))
            .join(silver_ipf_comp, "dt_competencia")
            .withColumn("desvio", F.abs(F.col("total_estimado") - F.col("total_silver")))
            .agg(F.max("desvio").alias("max_desvio"), F.sum("desvio").alias("soma_desvio"))
            .collect()[0]
        )
        assert desvio_ipf_comp["max_desvio"] <= 2, \
            f"Desvio mensal acima do esperado para puro arredondamento: {desvio_ipf_comp['max_desvio']}"
        print(f"Totais por competência: desvio máximo {desvio_ipf_comp['max_desvio']} exames "
              f"(soma {desvio_ipf_comp['soma_desvio']}) — efeito do arredondamento.")

        display(
            df_or_ipf.filter((F.col("municipio") == "Uruguaiana") & (F.col("ano") == 2021))
            .select("municipio", "mes_label", "quantidade_exames_estimada", "estimativa_fracionada")
            .orderBy("mes_num")
        )
else:
    print("Auditoria histórica desabilitada; dados publicados preservados.")

## 3. Perfil etário

`fato_conformidade_faixa_etaria` — município × ano × faixa. Proporções sobre exames com idade informada; grupos <25, 25–64 e ≥65 anos. Métricas repetidas por grupo não devem ser somadas diretamente.

In [0]:
df_faixa = (
    ler_silver("faixa_etaria")
    # Idade ignorada fica fora do denominador.
    .filter(F.col("faixa_etaria") != "Ignorado")
)

df_faixa_grupo = df_faixa.withColumn(
    "idade_limite_inferior",
    F.expr("try_cast(regexp_extract(faixa_etaria, '([0-9]+)', 1) as int)"),
).withColumn(
    "grupo_etario_diretriz",
    F.when(F.col("fl_populacao_alvo"), F.lit(GRUPO_ALVO))
     .when(F.col("idade_limite_inferior") < 25, F.lit(GRUPO_SOBRE))
     .otherwise(F.lit(GRUPO_SUB)),
)

win_muni_ano = Window.partitionBy("codigo_municipio", "ano")
win_grupo = Window.partitionBy("codigo_municipio", "ano", "grupo_etario_diretriz")

df_q3 = (
    df_faixa_grupo
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz")
    .agg(F.sum("quantidade").alias("quantidade_exames"))
    .withColumn("total_exames_muni_ano", F.sum("quantidade_exames").over(win_muni_ano))
    .withColumn(
        "proporcao_pct",
        percentual(F.col("quantidade_exames"), F.col("total_exames_muni_ano")),
    )
    .withColumn("quantidade_grupo", F.sum("quantidade_exames").over(win_grupo))
    .withColumn(
        "proporcao_grupo_pct",
        percentual(F.col("quantidade_grupo"), F.col("total_exames_muni_ano")),
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria",
        "grupo_etario_diretriz", "quantidade_exames", "total_exames_muni_ano",
        "proporcao_pct", "quantidade_grupo", "proporcao_grupo_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_preparar(
    df_q3,
    "fato_conformidade_faixa_etaria",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz"],
)

display(df_q3.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

### 3b. Indicador da população-alvo

`fato_indicador_populacao_alvo` — município × ano. Proporção = exames do grupo etário / total com idade informada × 100.

In [0]:
df_indicador_alvo = (
    df_q3
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(
        F.max("total_exames_muni_ano").alias("total_exames_informacao_etaria"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_ALVO, F.col("quantidade_grupo"))).alias("exames_populacao_alvo"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SOBRE, F.col("quantidade_grupo"))).alias("exames_sobrerastreamento"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SUB, F.col("quantidade_grupo"))).alias("exames_subrastreamento"),
    )
    .withColumn(
        "proporcao_populacao_alvo_pct",
        percentual(F.col("exames_populacao_alvo"), F.col("total_exames_informacao_etaria")),
    )
    .withColumn(
        "proporcao_sobrerastreamento_pct",
        percentual(F.col("exames_sobrerastreamento"), F.col("total_exames_informacao_etaria")),
    )
    .withColumn(
        "proporcao_subrastreamento_pct",
        percentual(F.col("exames_subrastreamento"), F.col("total_exames_informacao_etaria")),
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano",
        "exames_populacao_alvo", "exames_sobrerastreamento", "exames_subrastreamento",
        "total_exames_informacao_etaria",
        "proporcao_populacao_alvo_pct", "proporcao_sobrerastreamento_pct",
        "proporcao_subrastreamento_pct", "_silver_source", "_gold_processed_time",
    )
)

validar_e_preparar(
    df_indicador_alvo,
    "fato_indicador_populacao_alvo",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano"],
)

display(df_indicador_alvo.orderBy("codigo_municipio", "ano"))

### 3c. Razão anual de exames

`fato_evolucao_cobertura_municipios` — município × ano. Exames de 25–64 anos / população feminina de 25–64 anos do Censo IBGE 2022 × 100. Denominador fixo; procedimentos por 100 mulheres, não mulheres distintas ou cobertura efetiva. `taxa_cobertura_pct` é alias legado da razão anual.

In [0]:
df_exames_alvo = (
    ler_silver("faixa_etaria")
    .filter(F.col("fl_populacao_alvo") == True)
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(F.sum("quantidade").alias("total_exames_25_64"))
)

df_pop_alvo = (
    ler_gold("dim_populacao_municipios")
    .select(
        F.col("cd_ibge").cast("int").alias("codigo_municipio"),
        F.col("mulheres_25_64_anos"),
    )
)

df_evolucao_cobertura = (
    df_exames_alvo
    .join(df_pop_alvo, "codigo_municipio", "inner")
    .withColumn(
        "razao_anual_exames_pct",
        F.when(
            F.col("mulheres_25_64_anos") > 0,
            F.round((F.col("total_exames_25_64") / F.col("mulheres_25_64_anos")) * 100, 2),
        ),
    )
    # Alias legado para consumidores existentes.
    .withColumn("taxa_cobertura_pct", F.col("razao_anual_exames_pct"))
    .withColumn("meta_ans_pct", F.lit(META_ANUAL_PCT))
    .withColumn(
        "st_meta_ans",
        F.when(F.col("razao_anual_exames_pct") >= META_ANUAL_PCT, F.lit("🟢 Razão ≥33%"))
         .when(F.col("razao_anual_exames_pct") >= 3.0, F.lit("🟡 Razão de 3% a <33%"))
         .otherwise(F.lit("🔴 Razão <3%")),
    )
    .withColumn(
        "_silver_source",
        F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria + {CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios"),
    )
    .withColumn("_gold_processed_time", F.current_timestamp())
)

assert df_evolucao_cobertura.count() == 66, \
    f"Esperado 66 linhas (11 municípios × 6 anos); obtido {df_evolucao_cobertura.count()}"
assert df_evolucao_cobertura.select("codigo_municipio").distinct().count() == 11, \
    "Inner join com a dimensão populacional perdeu municípios da 10ª CRS"

validar_e_preparar(
    df_evolucao_cobertura,
    "fato_evolucao_cobertura_municipios",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "total_exames_25_64",
                      "mulheres_25_64_anos", "razao_anual_exames_pct"],
)

display(df_evolucao_cobertura.orderBy("codigo_municipio", "ano"))

### 3d. Referência anual de produção

`dim_meta_ans_municipios` — município. Referência exploratória = população feminina de 25–64 anos do Censo 2022 × 0,33, arredondada a inteiro. O patamar de 33% não garante cobertura individual em três anos.

In [0]:
df_meta_ans = (
    ler_gold("dim_populacao_municipios")
    .withColumn(
        "meta_ans_exames_anual",
        F.round(F.col("mulheres_25_64_anos") * (META_ANUAL_PCT / 100), 0).cast("long"),
    )
    .withColumn("meta_ans_pct", F.lit(META_ANUAL_PCT))
    .withColumn("codigo_municipio", F.col("cd_ibge").cast("int"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios (Censo IBGE 2022)"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "mulheres_25_64_anos",
        "meta_ans_exames_anual", "meta_ans_pct", "_silver_source", "_gold_processed_time",
    )
)

assert df_meta_ans.count() == 11, \
    f"Meta ANS deve ter 11 municípios da 10ª CRS; obtido {df_meta_ans.count()}"

validar_e_preparar(
    df_meta_ans,
    "dim_meta_ans_municipios",
    pk_cols=["codigo_municipio"],
    obrigatorio_cols=["codigo_municipio", "municipio", "mulheres_25_64_anos", "meta_ans_exames_anual"],
)

display(df_meta_ans.orderBy("codigo_municipio"))

## 4. Adequabilidade

`fato_adequabilidade_amostras` — município × ano × categoria. Proporção = quantidade da categoria / total de amostras classificadas × 100.

In [0]:
df_adeq = ler_silver("adequabilidade")

win_adeq_total = Window.partitionBy("codigo_municipio", "ano")

df_q4 = (
    df_adeq
    .groupBy("codigo_municipio", "municipio", "ano", "categoria_adequabilidade")
    .agg(F.sum("quantidade").alias("quantidade_amostras"))
    .withColumn("total_amostras_muni_ano", F.sum("quantidade_amostras").over(win_adeq_total))
    .withColumn(
        "proporcao_adequabilidade_pct",
        percentual(F.col("quantidade_amostras"), F.col("total_amostras_muni_ano")),
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "categoria_adequabilidade",
        "quantidade_amostras", "total_amostras_muni_ano", "proporcao_adequabilidade_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_preparar(
    df_q4,
    "fato_adequabilidade_amostras",
    pk_cols=["codigo_municipio", "ano", "categoria_adequabilidade"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "categoria_adequabilidade"],
)

display(df_q4.orderBy("codigo_municipio", "ano", "categoria_adequabilidade"))

## 5. Exames alterados

`fato_exames_alterados` — município × ano × faixa. Proporção operacional = alterados / válidos × 100; exclui amostras insatisfatórias e rejeitadas.

O cruzamento adequabilidade × idade não está disponível: as exclusões observadas por município/ano são alocadas proporcionalmente às faixas, por divisão inteira cumulativa. O denominador etário é estimado; a soma das exclusões é preservada. O full outer join mantém faixas omitidas no relatório de alterados. Proporções agregadas usam razão de somas, nunca média de percentuais.

In [0]:
df_resultados = ler_silver("resultados_citopatologicos")
df_exames_totais = ler_silver("faixa_etaria")
df_adequabilidade = ler_silver("adequabilidade")

df_alterados = (
    df_resultados
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("quantidade_alterados"))
)

df_denominador = (
    df_exames_totais
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("total_exames_faixa"))
)

# Exclui insatisfatórias e rejeitadas do denominador.
df_exclusoes = (
    df_adequabilidade
    .filter(F.col("categoria_adequabilidade").isin("Insatisfatoria", "Rejeitada"))
    .groupBy("codigo_municipio", "ano")
    .agg(F.sum("quantidade").alias("qtd_excluidas_muni_ano"))
)

# Alocação cumulativa inteira conserva as exclusões por município/ano.
win_faixa = Window.partitionBy("codigo_municipio", "ano").orderBy("faixa_etaria")
df_denominador = (
    df_denominador
    .withColumn("total_exames_muni_ano",
        F.sum("total_exames_faixa").over(win_faixa.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))
    .join(df_exclusoes, ["codigo_municipio", "ano"], how="left")
    .withColumn("qtd_excluidas_muni_ano", F.coalesce(F.col("qtd_excluidas_muni_ano"), F.lit(0)))
    # Divisão inteira evita perda por arredondamento de ponto flutuante.
    .withColumn("_quota_num", F.col("total_exames_faixa") * F.col("qtd_excluidas_muni_ano"))
    .withColumn("_cum_hi", F.sum("_quota_num").over(win_faixa.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn("_cum_lo", F.coalesce(F.lag("_cum_hi", 1).over(win_faixa), F.lit(0)))
    .withColumn("excluidas_faixa",
        F.coalesce(
            F.least(
                F.expr("_cum_hi div total_exames_muni_ano") - F.expr("_cum_lo div total_exames_muni_ano"),
                F.col("total_exames_faixa")),
            F.lit(0)))
    .withColumn("total_exames_validos", F.col("total_exames_faixa") - F.col("excluidas_faixa"))
    .drop("_quota_num", "_cum_hi", "_cum_lo", "total_exames_muni_ano")
)

# Full outer preserva faixas sem registros de alterados.
df_q5 = (
    df_alterados.alias("a")
    .join(df_denominador.alias("d"), on=["codigo_municipio", "ano", "faixa_etaria"], how="full_outer")
    .select(
        F.col("codigo_municipio"),
        F.coalesce(F.col("a.municipio"), F.col("d.municipio")).alias("municipio"),
        F.col("ano"),
        F.col("faixa_etaria"),
        F.coalesce(F.col("a.quantidade_alterados"), F.lit(0)).alias("quantidade_alterados"),
        F.col("d.total_exames_faixa"),
        F.coalesce(F.col("d.qtd_excluidas_muni_ano"), F.lit(0)).alias("qtd_excluidas_muni_ano"),
        F.coalesce(F.col("d.excluidas_faixa"), F.lit(0)).alias("excluidas_faixa"),
        F.col("d.total_exames_validos"),
    )
    .withColumn(
        "proporcao_alterados_pct",
        percentual(F.col("quantidade_alterados"), F.col("total_exames_validos")),
    )
    .withColumn("grupo_etario_3", grupo_etario_3())
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(
        f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos + {CATALOG}.{SCHEMA_SILVER}.faixa_etaria + {CATALOG}.{SCHEMA_SILVER}.adequabilidade"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria", "grupo_etario_3",
        "quantidade_alterados", "total_exames_faixa", "qtd_excluidas_muni_ano",
        "excluidas_faixa", "total_exames_validos", "proporcao_alterados_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_preparar(
    df_q5,
    "fato_exames_alterados",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria"],
)

display(df_q5.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

### 5b/5c. Categorias citopatológicas

`fato_categoria_anual`: município × categoria × ano (528 linhas); fontes mensais e anuais disjuntas evitam dupla contagem.

`fato_categoria_faixa_etaria`: município × categoria × faixa (1.320 linhas, consolidado 2020–2025); não há cruzamento observado categoria × faixa × ano. Grades completadas com zero para categorias omitidas pelo TABNET. Totais repetidos não são aditivos.

O fato de alterados permanece o indicador canônico: extrações independentes divergem por município/ano mesmo com fechamento regional igual.

In [0]:
df_cat_base = (
    ler_silver("resultados_categoria_anual")
    .select("codigo_municipio", "municipio", "categoria", "ano", "quantidade")
    .unionByName(
        ler_silver("resultados_categoria_mensal")
        .select("codigo_municipio", "municipio", "categoria", "ano", "quantidade")
    )
)

df_obs_anual = (
    df_cat_base
    .groupBy("codigo_municipio", "municipio", "categoria", "ano")
    .agg(F.sum("quantidade").alias("quantidade_alterados"))
)

dim_mun = df_cat_base.select("codigo_municipio", "municipio").distinct()
dim_cat = df_cat_base.select("categoria").distinct()
dim_ano = spark.sql(f"SELECT explode(sequence({ANO_INICIO}, {ANO_FIM})) AS ano")

# Zero-fill apenas para categorias omitidas no TABNET.
grade_anual = dim_mun.crossJoin(dim_cat).crossJoin(dim_ano)

win_muni_ano = Window.partitionBy("codigo_municipio", "ano")
df_q5b = (
    grade_anual
    .join(df_obs_anual, ["codigo_municipio", "municipio", "categoria", "ano"], "left")
    .withColumn("quantidade_alterados", F.coalesce(F.col("quantidade_alterados"), F.lit(0)))
    .withColumn("total_alterados_muni_ano", F.sum("quantidade_alterados").over(win_muni_ano))
    .withColumn(
        "proporcao_categoria_pct",
        percentual(F.col("quantidade_alterados"), F.col("total_alterados_muni_ano")),
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(
        f"{CATALOG}.{SCHEMA_SILVER}.resultados_categoria_anual + {CATALOG}.{SCHEMA_SILVER}.resultados_categoria_mensal"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "categoria", "ano",
        "quantidade_alterados", "total_alterados_muni_ano", "proporcao_categoria_pct",
        "_silver_source", "_gold_processed_time",
    )
)

assert dim_cat.count() == 8, "Esperadas 8 categorias"
assert dim_mun.count() == 11, "Esperados 11 municípios"
assert df_q5b.count() == 11 * 8 * 6, "Grade 11×8×6 incompleta"
assert df_q5b.agg(F.sum("quantidade_alterados")).collect()[0][0] == 4479, "Fechamento ≠ 4.479"

# Fontes mensal e anual devem ser disjuntas.
inter_5b = (
    ler_silver("resultados_categoria_anual")
    .select("codigo_municipio", "categoria", "ano")
    .intersect(
        ler_silver("resultados_categoria_mensal")
        .select("codigo_municipio", "categoria", "ano")
    )
    .count()
)
assert inter_5b == 0, f"{inter_5b} pares município/categoria/ano nas DUAS Silver — dupla contagem"

validar_e_preparar(
    df_q5b,
    "fato_categoria_anual",
    pk_cols=["codigo_municipio", "categoria", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "categoria", "ano"],
)

display(df_q5b.orderBy("codigo_municipio", "ano", "categoria").limit(20))

In [0]:
df_faixa_base = ler_silver("resultados_categoria_faixa_etaria")

df_obs_faixa = (
    df_faixa_base
    .groupBy("codigo_municipio", "municipio", "categoria", "faixa_etaria")
    .agg(F.sum("quantidade").alias("quantidade_alterados"))
)

dim_faixa = df_faixa_base.select("faixa_etaria").distinct()

# Completa a grade; totais repetidos não são aditivos.
grade_faixa = dim_mun.crossJoin(dim_cat).crossJoin(dim_faixa)

win_muni_cat = Window.partitionBy("codigo_municipio", "categoria")
df_q5c = (
    grade_faixa
    .join(df_obs_faixa, ["codigo_municipio", "municipio", "categoria", "faixa_etaria"], "left")
    .withColumn("quantidade_alterados", F.coalesce(F.col("quantidade_alterados"), F.lit(0)))
    .withColumn("grupo_etario_3", grupo_etario_3())
    .withColumn("fl_populacao_alvo", F.col("faixa_etaria").isin(*FAIXAS_25_64))
    .withColumn("total_alterados_muni_categoria", F.sum("quantidade_alterados").over(win_muni_cat))
    .withColumn(
        "proporcao_faixa_pct",
        percentual(F.col("quantidade_alterados"), F.col("total_alterados_muni_categoria")),
    )
    .withColumn("cd_ibge", codigo_ibge())
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.resultados_categoria_faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "categoria", "faixa_etaria",
        "grupo_etario_3", "fl_populacao_alvo", "quantidade_alterados",
        "total_alterados_muni_categoria", "proporcao_faixa_pct",
        "_silver_source", "_gold_processed_time",
    )
)

assert dim_faixa.count() == 15, "Esperadas 15 faixas"
assert df_q5c.count() == 11 * 8 * 15, "Grade 11×8×15 incompleta"
assert df_q5c.agg(F.sum("quantidade_alterados")).collect()[0][0] == 4479, "Fechamento ≠ 4.479"
assert df_q5c.filter(F.col("grupo_etario_3") == "Ignorado").count() == 0, "Faixa sem mapeamento de grupo"

validar_e_preparar(
    df_q5c,
    "fato_categoria_faixa_etaria",
    pk_cols=["codigo_municipio", "categoria", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "categoria", "faixa_etaria", "grupo_etario_3"],
)

coer = (
    df_q5b.groupBy("codigo_municipio", "categoria")
    .agg(F.sum("quantidade_alterados").alias("t_anual"))
    .join(
        df_q5c.groupBy("codigo_municipio", "categoria")
        .agg(F.sum("quantidade_alterados").alias("t_faixa")),
        ["codigo_municipio", "categoria"], "full_outer",
    )
    .filter(F.col("t_anual").isNull() | F.col("t_faixa").isNull() | (F.col("t_anual") != F.col("t_faixa")))
    .count()
)
assert coer == 0, f"Coerência fato_categoria_anual × fato_categoria_faixa_etaria: {coer} divergências"
print("Coerência gold: 0 divergências em 88 pares município × categoria.")

display(df_q5c.orderBy("codigo_municipio", "categoria", "faixa_etaria").limit(20))

## 6. Validação integrada e persistência

Conferência de chaves, proporções, alocação de exclusões, fechamento e coerência dos fatos. A regressão compara colunas, tipos e registros analíticos publicados, exceto metadados técnicos.

Somente após todas as validações ocorre a gravação dos dez fatos/dimensões. Cada overwrite é transacional por tabela, não em conjunto. Mantenha `VALIDAR_REGRESSAO = True` até aprovar uma atualização intencional.

Para executar somente o pipeline, encerre após a seção 6. As demais seções não gravam tabelas.

In [0]:
tabelas_gold = [
    "fato_exames_temporais_territoriais",
    "fato_outubro_rosa_producao_municipio_ano",
    "fato_conformidade_faixa_etaria",
    "fato_indicador_populacao_alvo",
    "fato_adequabilidade_amostras",
    "fato_exames_alterados",
    "fato_categoria_anual",
    "fato_categoria_faixa_etaria",
    "fato_evolucao_cobertura_municipios",
    "dim_meta_ans_municipios",
]

exigir(set(tabelas_gold) == set(fatos_preparados), "Fatos preparados incompletos")
for tabela in tabelas_gold:
    print(f"{tabela}: {contagens_preparadas[tabela]} linhas preparadas")

df_g1 = ler_gold("fato_exames_temporais_territoriais")
soma_part = (
    df_g1.groupBy("dt_competencia")
    .agg(F.round(F.sum("participacao_territorial_pct"), 1).alias("soma_participacao"))
)
err_part = soma_part.filter(~F.col("soma_participacao").between(99.5, 100.5)).count()
print(f"\nQ1/Q2 — competências fora de [99.5, 100.5]% de participação somada: {err_part}")

df_g3 = ler_gold("fato_conformidade_faixa_etaria")
err_q3 = (
    df_g3.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q3 — pares município/ano fora de [99.5, 100.5]%: {err_q3}")

df_ind = ler_gold("fato_indicador_populacao_alvo")
err_ind = (
    df_ind.withColumn(
        "soma",
        F.round(
            F.col("proporcao_populacao_alvo_pct")
            + F.col("proporcao_sobrerastreamento_pct")
            + F.col("proporcao_subrastreamento_pct"), 1),
    )
    .filter(~F.col("soma").between(99.5, 100.5))
    .count()
)
print(f"Q3-KPI — município/ano fora de [99.5, 100.5]%: {err_ind}")

df_g4 = ler_gold("fato_adequabilidade_amostras")
err_q4 = (
    df_g4.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_adequabilidade_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q4 — pares município/ano fora de [99.5, 100.5]%: {err_q4}")

df_g5 = ler_gold("fato_exames_alterados")
df_adeq_s = ler_silver("adequabilidade")

err_q5_aloc = (
    df_g5.groupBy("codigo_municipio", "ano")
    .agg(F.sum("excluidas_faixa").alias("alocadas"), F.max("qtd_excluidas_muni_ano").alias("declaradas"))
    .filter(F.col("alocadas") != F.col("declaradas"))
    .count()
)
print(f"Q5 (a) — pares município/ano com alocação de exclusões divergente: {err_q5_aloc}")

tot_q5 = df_g5.agg(
    F.sum("total_exames_faixa").alias("faixa"),
    F.sum("excluidas_faixa").alias("excluidas"),
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
).collect()[0]
sat_silver = (
    df_adeq_s.filter(F.col("categoria_adequabilidade") == "Satisfatoria")
    .agg(F.sum("quantidade")).collect()[0][0]
)
err_q5_fech = int((tot_q5["faixa"] - tot_q5["excluidas"]) != tot_q5["validos"]) + \
              int(tot_q5["validos"] != sat_silver)
print(f"Q5 (b) — total_exames_faixa ({tot_q5['faixa']:,}) − excluidas ({tot_q5['excluidas']:,}) "
      f"= validos ({tot_q5['validos']:,}) | satisfatórias Silver: {sat_silver:,} "
      f"| alterados: {tot_q5['alterados']:,} | divergências: {err_q5_fech}")

err_q5_neg = df_g5.filter(
    (F.col("total_exames_validos") < 0)
    | ((F.col("total_exames_validos").isNotNull())
       & (F.col("total_exames_validos") < F.col("quantidade_alterados")))
).count()
print(f"Q5 (c) — linhas com denominador negativo ou < alterados: {err_q5_neg}")

df_g5b = ler_gold("fato_categoria_anual")
df_g5c = ler_gold("fato_categoria_faixa_etaria")
soma_5b = df_g5b.agg(F.sum("quantidade_alterados")).collect()[0][0]
soma_5c = df_g5c.agg(F.sum("quantidade_alterados")).collect()[0][0]
err_q5cat = (
    int(soma_5b != 4479)
    + int(soma_5c != 4479)
    + df_g5b.groupBy("codigo_municipio", "categoria")
        .agg(F.sum("quantidade_alterados").alias("t_anual"))
        .join(
            df_g5c.groupBy("codigo_municipio", "categoria")
            .agg(F.sum("quantidade_alterados").alias("t_faixa")),
            ["codigo_municipio", "categoria"], "full_outer",
        )
        .filter(F.col("t_anual").isNull() | F.col("t_faixa").isNull() | (F.col("t_anual") != F.col("t_faixa")))
        .count()
)
print(f"Q5-categorias — soma anual: {soma_5b:,} | soma faixa: {soma_5c:,} | "
      f"divergências de coerência mun×categoria: {err_q5cat}")

df_or = ler_gold("fato_outubro_rosa_producao_municipio_ano")
erro_or = df_or.filter(
    (F.col("meses_observados") < 0)
    | (F.col("meses_observados") > 4)
    | ((F.col("meses_observados") < 4) & F.col("exames_set_dez").isNotNull())
    | ((F.col("meses_observados") == 4) & F.col("exames_set_dez").isNull())
).count()
assert erro_or == 0, "Outubro Rosa: meses observados e total set–dez inconsistentes"
print("Outubro Rosa: completude temporal validada.")

escopo = df_g1.agg(
    F.countDistinct("codigo_municipio").alias("municipios"),
    F.min("ano").alias("ano_min"),
    F.max("ano").alias("ano_max"),
)
display(escopo)

assert (err_part == 0 and err_q3 == 0 and err_ind == 0 and err_q4 == 0
        and err_q5_aloc == 0 and err_q5_fech == 0 and err_q5_neg == 0
        and err_q5cat == 0), \
    "Validação final falhou — verifique as contagens acima."
escopo_val = escopo.first()
exigir(
    escopo_val["municipios"] == 11
    and escopo_val["ano_min"] == ANO_INICIO
    and escopo_val["ano_max"] == ANO_FIM,
    "Escopo territorial ou temporal inválido",
)

numerador_silver = ler_silver("resultados_citopatologicos").agg(F.sum("quantidade")).first()[0]
exigir(
    tot_q5["alterados"] == numerador_silver,
    "Numerador de exames alterados diverge da Silver",
)

gravar_fatos()
for tabela, linhas in contagens_preparadas.items():
    exigir(
        spark.table(nome_tabela(SCHEMA_GOLD, tabela)).count() == linhas,
        f"{tabela}: contagem publicada divergente",
    )
print("Validação e gravação da camada Gold concluídas.")
fatos_preparados.clear()

## 7. Sazonalidade — somente leitura

Índice = média do mês / média das competências observadas × 100. Agosto/2022 ausente: cinco ocorrências de agosto e seis dos demais meses (71 competências). O índice municipal usa a média do próprio município.

In [0]:
%sql
WITH mes_ano AS (
  SELECT ano, mes, SUM(quantidade) AS qtd
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY ano, mes
),
base AS (
  SELECT AVG(qtd) AS media_geral FROM mes_ano
)
SELECT
  m.mes,
  CASE m.mes
    WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Marco' WHEN 4 THEN 'Abril'
    WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho' WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto'
    WHEN 9 THEN 'Setembro' WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
  END AS nome_mes,
  COUNT(*) AS ano_mes_disponiveis,
  ROUND(AVG(m.qtd), 1) AS media_mensal,
  ROUND(AVG(m.qtd) / b.media_geral * 100, 1) AS indice_sazonal,
  ROUND((AVG(m.qtd) / b.media_geral - 1) * 100, 1) AS desvio_pct_vs_media,
  RANK() OVER (ORDER BY AVG(m.qtd) ASC) AS rank_mais_fraco
FROM mes_ano m CROSS JOIN base b
GROUP BY m.mes, b.media_geral
ORDER BY m.mes

In [0]:
%sql
WITH mes_ano AS (
  SELECT ano, mes, SUM(quantidade) AS qtd
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY ano, mes
)
SELECT
  mes,
  MAX(CASE WHEN ano = 2020 THEN qtd END) AS a2020,
  MAX(CASE WHEN ano = 2021 THEN qtd END) AS a2021,
  MAX(CASE WHEN ano = 2022 THEN qtd END) AS a2022,
  MAX(CASE WHEN ano = 2023 THEN qtd END) AS a2023,
  MAX(CASE WHEN ano = 2024 THEN qtd END) AS a2024,
  MAX(CASE WHEN ano = 2025 THEN qtd END) AS a2025,
  ROUND(AVG(qtd), 1) AS media
FROM mes_ano
GROUP BY mes
ORDER BY mes

In [0]:
%sql
WITH mun_mes AS (
  SELECT codigo_municipio, municipio, mes,
         SUM(quantidade) AS qtd_total,
         COUNT(DISTINCT ano) AS n_anos
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY 1, 2, 3
),
mun_media AS (
  SELECT codigo_municipio, municipio, AVG(qtd_total / n_anos) AS media_mes_muni
  FROM mun_mes
  GROUP BY 1, 2
),
idx AS (
  SELECT m.codigo_municipio, m.municipio, m.mes,
         m.qtd_total / m.n_anos AS media_mensal,
         (m.qtd_total / m.n_anos) / g.media_mes_muni * 100 AS indice
  FROM mun_mes m
  JOIN mun_media g USING (codigo_municipio, municipio)
)
SELECT
  municipio,
  MAX(CASE WHEN rank_asc = 1 THEN CONCAT(nome, ' (', ROUND(indice, 0), ')') END) AS mes_mais_fraco,
  ROUND(AVG(CASE WHEN mes IN (5, 6, 7, 8, 9) THEN indice END), 1) AS indice_medio_maio_set,
  ROUND(AVG(CASE WHEN mes IN (11, 12) THEN indice END), 1) AS indice_medio_nov_dez,
  MAX(CASE WHEN rank_desc = 1 THEN CONCAT(nome, ' (', ROUND(indice, 0), ')') END) AS mes_mais_forte,
  ROUND(SUM(media_mensal), 0) AS media_anual
FROM (
  SELECT *,
    DENSE_RANK() OVER (PARTITION BY municipio ORDER BY indice ASC) AS rank_asc,
    DENSE_RANK() OVER (PARTITION BY municipio ORDER BY indice DESC) AS rank_desc,
    CASE mes
      WHEN 1 THEN 'Jan' WHEN 2 THEN 'Fev' WHEN 3 THEN 'Mar' WHEN 4 THEN 'Abr'
      WHEN 5 THEN 'Mai' WHEN 6 THEN 'Jun' WHEN 7 THEN 'Jul' WHEN 8 THEN 'Ago'
      WHEN 9 THEN 'Set' WHEN 10 THEN 'Out' WHEN 11 THEN 'Nov' WHEN 12 THEN 'Dez'
    END AS nome
  FROM idx
)
GROUP BY municipio
ORDER BY indice_medio_nov_dez DESC

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

total = (
    ler_silver("exames_geral")
    .agg(F.sum("quantidade").alias("total"))
    .collect()[0]["total"]
)
print(f"Total de exames 2020-2025: {total:,} (esperado: 131.058)")
assert int(total) == 131058, f"Total divergente do esperado: {total}"

df_sazon = (
    ler_silver("exames_geral")
    .groupBy("ano", "mes")
    .agg(F.sum("quantidade").alias("qtd"))
)
media_geral = df_sazon.agg(F.avg("qtd").alias("mg")).collect()[0]["mg"]

df_sazon_plot = (
    df_sazon.groupBy("mes")
    .agg(F.round(F.avg("qtd") / F.lit(media_geral) * 100, 1).alias("indice_sazonal"))
    .withColumn(
        "nome_mes",
        F.expr("""
          CASE mes
            WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Marco' WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho' WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto'
            WHEN 9 THEN 'Setembro' WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
          END
        """),
    )
    .orderBy("mes")
    .toPandas()
)

fraca = df_sazon_plot.loc[df_sazon_plot["mes"].between(5, 9), "indice_sazonal"].mean()
pico = df_sazon_plot.loc[df_sazon_plot["mes"].isin([11, 12]), "indice_sazonal"].mean()
print(f"Janela fraca (mai-set): índice médio {fraca:.1f} | Pico (nov-dez): índice médio {pico:.1f} | razão ≈ {pico / fraca:.1f}x")

cores = ["#d9534f" if 5 <= m <= 9 else ("#5cb85c" if m in (11, 12) else "#adb5bd") for m in df_sazon_plot["mes"]]
fig, ax = plt.subplots(figsize=(11, 5))
barras = ax.bar(df_sazon_plot["nome_mes"], df_sazon_plot["indice_sazonal"], color=cores, edgecolor="black", linewidth=0.5)
ax.axhline(100, color="black", linestyle="--", linewidth=1, label="Mês médio (índice 100)")
for b, v in zip(barras, df_sazon_plot["indice_sazonal"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 3, f"{v:.0f}", ha="center", fontsize=9)
ax.set_title("Índice sazonal mensal — exames citopatológicos, 10ª CRS (2020-2025)")
ax.set_ylabel("Índice (média mensal geral = 100)")
ax.set_ylim(0, 185)
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 7.5 Síntese sazonal

Resultados históricos: maio–setembro apresentam menores índices (76,9–82,7); novembro–dezembro, os maiores (157,6–162,6). Investigar disponibilidade de serviços e datas de processamento antes de propor alterações de capacidade.

## 8. Exames alterados — análise exploratória (somente leitura)

Referência regional: 130.598 exames analisáveis (131.058 − 390 insatisfatórias − 70 rejeitadas) e 4.479 alterados. As análises etárias excluem `Ignorado` e `Até 9 anos`; taxas agregadas usam razão de somas. O modelo binomial e a projeção são exploratórios, com dados agregados e denominadores etários alocados.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

df = spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados")

FAIXAS_EXCLUIR = ["Ignorado", "Até 9 anos"]  # base ínfima / fora da diretriz

n_rows = df.count()
# A referência regional usa o mesmo universo das análises municipais.
tot = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .select(
        F.sum("total_exames_validos").alias("validos"),
        F.sum("quantidade_alterados").alias("alterados"),
    )
    .collect()[0]
)

print(f"Linhas (município × ano × faixa): {n_rows}")
print(f"Total de exames válidos: {tot['validos']:,}")
print(f"Total de exames alterados: {tot['alterados']:,}")
print(f"Taxa global ponderada: {tot['alterados'] / tot['validos'] * 100:.3f}%")
print("\nCobertura e volume por ano:")
df.select(
    F.countDistinct("municipio").alias("municipios"),
    F.countDistinct("ano").alias("anos"),
    F.countDistinct("faixa_etaria").alias("faixas"),
).show()
df.groupBy("ano").agg(
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
    F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa_pct"),
).orderBy("ano").show()

In [0]:
taxa_faixa = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria")
    .agg(
        F.sum("quantidade_alterados").alias("alterados"),
        F.sum("total_exames_validos").alias("validos"),
    )
    .withColumn("taxa_pct", F.round(F.col("alterados") / F.col("validos") * 100, 2))
    .orderBy(F.desc("taxa_pct"))
)
display(taxa_faixa)

tf = taxa_faixa.toPandas()
top = tf.iloc[0]
print(f">>> Maior taxa de alteração: {top['faixa_etaria']} — {top['taxa_pct']}% "
      f"({top['alterados']:.0f} alterados de {top['validos']:.0f} exames)")

In [0]:
pivot = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria", "ano")
    .agg(F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa"))
    .toPandas()
    .pivot(index="faixa_etaria", columns="ano", values="taxa")
)

ordem = ["10 a 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
         "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos",
         "60 a 64 anos", "65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos"]
pivot = pivot.reindex([f for f in ordem if f in pivot.index])

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(pivot.values.astype(float), cmap="OrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)), [str(int(c)) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.tick_params(labelsize=8)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Taxa de alterados (%)")
ax.set_title("Taxa de exames alterados por faixa etária e ano — 10ª CRS (2020–2025)")
plt.tight_layout()
plt.show()

In [0]:
anual = (
    df.groupBy("ano")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .orderBy("ano")
    .toPandas()
)
anual["taxa"] = anual.alt / anual.val * 100

b1, b0 = np.polyfit(anual.ano, anual.taxa, 1)  # MQO sobre 6 pontos anuais
print(anual[["ano", "alt", "val", "taxa"]].round(3).to_string(index=False))
ss_res = ((anual.taxa - (b0 + b1 * anual.ano)) ** 2).sum()
r2 = 1 - ss_res / ((anual.taxa - anual.taxa.mean()) ** 2).sum()
print(f"\nTendência linear: {b1:+.3f} p.p./ano (R² = {r2:.2f} — descritiva; a série tem forma em V, "
      "a quantificação defensável do efeito ano está na 8.7)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(anual.ano, anual.taxa, marker="o", label="Taxa observada")
ax.plot(anual.ano, b0 + b1 * anual.ano, "--", label="Tendência linear")
for _, r in anual.iterrows():
    ax.annotate(f"{r.taxa:.2f}%", (r.ano, r.taxa), textcoords="offset points", xytext=(0, 7), fontsize=8)
ax.set_xlabel("Ano"); ax.set_ylabel("Taxa (%)")
ax.set_title("Taxa anual de exames alterados — 10ª CRS")
ax.legend()
plt.tight_layout(); plt.show()

In [0]:
mun = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("municipio")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .toPandas()
)
mun["taxa"] = mun.alt / mun.val * 100

mun[["ic_lo", "ic_hi"]] = mun.apply(lambda r: wilson(r["alt"], r["val"]), axis=1, result_type="expand")
mun = mun.sort_values("taxa", ascending=False).reset_index(drop=True)
print(mun.round(2).to_string(index=False))

rk_v, rk_t = mun["val"].rank(), mun["taxa"].rank()
rho = np.corrcoef(rk_v, rk_t)[0, 1]
print(f"\nSpearman volume × taxa (municípios): rho = {rho:+.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(mun.taxa, range(len(mun)),
            xerr=[mun.taxa - mun.ic_lo, mun.ic_hi - mun.taxa],
            fmt="o", capsize=3)
ax.set_yticks(range(len(mun)), mun.municipio, fontsize=8)
ax.axvline(tot["alterados"] / tot["validos"] * 100, color="gray", ls=":",
           label=f"Taxa regional {tot['alterados'] / tot['validos'] * 100:.2f}%")
ax.set_xlabel("Taxa (%) com IC 95% de Wilson")
ax.set_title("Taxa de alterados por município (2020–2025)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [0]:
def grupo(faixa: str) -> str:
    if any(faixa.startswith(p) for p in ["25 a 29", "30 a 34", "35 a 39", "40 a 44",
                                          "45 a 49", "50 a 54", "55 a 59", "60 a 64"]):
        return "25–64 (população-alvo)"
    if faixa.startswith(("10 a 14", "15 a 19", "20 a 24")):
        return "<25 (sobrerastreamento)"
    return "65+ (sub-rastreamento)"

g = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
g["grupo"] = g["faixa_etaria"].map(grupo)
res = g.groupby("grupo").agg(alt=("quantidade_alterados", "sum"),
                              val=("total_exames_validos", "sum"))
res["taxa_pct"] = (res.alt / res.val * 100).round(2)
print(res.sort_values("taxa_pct", ascending=False).to_string())
print(f"\nSoma das taxas ≠ 100% pois a taxa é de ALTERADOS (não a distribuição de exames).")

In [0]:
IDADE = {"10 a 14 anos": 12.5, "15 a 19 anos": 17.5, "20 a 24 anos": 22.5, "25 a 29 anos": 27.5,
         "30 a 34 anos": 32.5, "35 a 39 anos": 37.5, "40 a 44 anos": 42.5, "45 a 49 anos": 47.5,
         "50 a 54 anos": 52.5, "55 a 59 anos": 57.5, "60 a 64 anos": 62.5, "65 a 69 anos": 67.5,
         "70 a 74 anos": 72.5, "75 a 79 anos": 77.5, "Acima de 79 anos": 82.5}

m = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
m["idade"] = m["faixa_etaria"].map(IDADE)
m = m.dropna(subset=["idade"]).copy()
# IRLS binomial agrupado: exclui denominadores ausentes ou zero.
n_antes = len(m)
m = m[m["total_exames_validos"].notna() & (m["total_exames_validos"] > 0)
      & m["quantidade_alterados"].notna()].copy()
print(f"Linhas usadas no modelo: {len(m)} (descartadas {n_antes - len(m)} com denominador NULL/0)")

def fit_binomial(X, k, n, iters=200, tol=1e-10):
    """Regressão logística ponderada sobre proporções (IRLS).
    Retorna beta, covariância e flag de convergência."""
    beta = np.zeros(X.shape[1])
    converged = False
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30)
        mu = 1 / (1 + np.exp(-eta))
        w = n * mu * (1 - mu) + 1e-9
        z = eta + (k / n - mu) / np.maximum(mu * (1 - mu), 1e-9)
        beta_new = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * z))
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            converged = True
            break
        beta = beta_new
    eta = np.clip(X @ beta, -30, 30)
    mu = 1 / (1 + np.exp(-eta))
    w = n * mu * (1 - mu) + 1e-9
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    return beta, cov, converged

idade_c = (m["idade"] - 47.5) / 10   # idade centralizada: OR por década a partir de 47,5 anos
ano_c = m["ano"] - 2022.5            # ano centralizado na série
X = np.column_stack([np.ones(len(m)), idade_c, ano_c])
beta, cov, converged = fit_binomial(X, m["quantidade_alterados"].astype(float).values,
                                     m["total_exames_validos"].astype(float).values)
assert converged, "IRLS não convergiu em 200 iterações — coeficientes não confiáveis."
print("IRLS convergiu (tol 1e-10).")

se = np.sqrt(np.diag(cov))
names = ["intercepto (taxa aos 47,5 anos, ano central)", "idade (por década)", "ano (tendência)"]
print("Coeficientes (log-odds) e razão de chances (OR) com IC 95%:")
for nm, b, s in zip(names, beta, se):
    lo, hi = np.exp(b - 1.96 * s), np.exp(b + 1.96 * s)
    print(f"  {nm:<42} b={b:+.4f} (ep={s:.4f})   OR={np.exp(b):.3f}  IC95%[{lo:.3f}; {hi:.3f}]")

faixas_unicas = sorted(m["faixa_etaria"].unique(), key=lambda f: IDADE[f])
pred = pd.DataFrame({"faixa": faixas_unicas, "idade": [IDADE[f] for f in faixas_unicas]})
xx = np.column_stack([np.ones(len(pred)), (pred["idade"] - 47.5) / 10, np.zeros(len(pred))])
pred["p_prevista_pct"] = 100 / (1 + np.exp(-(xx @ beta)))
obs = taxa_faixa.toPandas().set_index("faixa_etaria")["taxa_pct"]
pred["p_observada_pct"] = pred["faixa"].map(obs)
print("\nProbabilidade prevista vs observada por faixa:")
print(pred.round(2).to_string(index=False))

In [0]:
for alvo in (2026,):
    x = np.array([1.0, 0.0, alvo - 2022.5])   # idade na média; efeito do ano
    p = 1 / (1 + np.exp(-(x @ beta)))
    var = float(x @ cov @ x)
    se_p = p * (1 - p) * np.sqrt(var)
    print(f"Taxa prevista para {alvo}: {p * 100:.2f}%  "
          f"(IC95% da MÉDIA prevista, método delta ≈ "
          f"[{(p - 1.96 * se_p) * 100:.2f}%; {(p + 1.96 * se_p) * 100:.2f}%] — não é intervalo de previsão)")

print("\nExtrapolação com apenas 6 pontos anuais e dados AGREGADOS (sem variáveis individuais).")
print("   Usar como indicador de TENDÊNCIA para planejamento, não como previsão clínica.")

In [0]:
q5_muni_ano = (
    spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados")
    .groupBy("municipio", "ano")
    .agg(F.sum("quantidade_alterados").alias("alterados"))
)

pivot_muni_ano = (
    q5_muni_ano.groupBy("municipio")
    .pivot("ano", [2020, 2021, 2022, 2023, 2024, 2025])
    .agg(F.sum("alterados"))
    .toPandas()
    .set_index("municipio")
)
pivot_muni_ano["Total"] = pivot_muni_ano.sum(axis=1)
pivot_muni_ano.loc["Total"] = pivot_muni_ano.sum()
print("Exames alterados por município e ano (10ª CRS, 2020–2025):")
print(pivot_muni_ano.fillna(0).astype(int).to_string())

### 8.10 Síntese — o que se pode extrair desta tabela — resultados históricos

**Achados (reexecução de 03/nov/2026 — denominador = 130.598 exames analisáveis, excluindo 390 insatisfatórias e 70 rejeitadas):**
* Taxa global: **3,43%** (4.479 alterados / 130.598 exames analisáveis).
* Faixa com maior taxa: **10-14 anos (4,91%)** — porém base minúscula (8/163). Entre faixas robustas: **15-19 (4,40%)** e **35-39 (4,14%)**; na população-alvo, **25-29 (3,92%)** lidera.
* Padrão etário: a taxa **decresce com a idade** — modelo binomial: OR = **0,889 por década** de idade (IC95% 0,870–0,907), ou ~11% menos chance de resultado alterado a cada década.
* Evolução anual: 3,09% (2020) → pico **4,72% (2023)** → queda a **1,74% (2025)**; tendência −0,22 p.p./ano; efeito ano no modelo: OR = 0,927 (−7% ao ano).
* Grupos da diretriz: **<25 = 4,01%** > 25–64 = 3,43% > 65+ = 2,55% — o sobrerastreamento concentra a maior taxa, reforçando a diretriz de não rastrear <25 anos.
* Municípios: **Quaraí (4,60%)** e **Uruguaiana (4,55%)** no topo; **Sant'Ana do Livramento (1,69%)** e **São Gabriel (2,35%)** na base. IC de Wilson mostra que Maçambará (IC 1,88–6,62) e Santa Margarida (3,10–5,95) são estatisticamente imprecisos (poucos exames).
* Correlação volume × taxa (Spearman): **rho = −0,12** — não há relação entre volume de exames e taxa de alteração.
* Projeção ilustrativa 2026 (efeito ano): **2,53%** (IC95% da **média prevista**, método delta: 2,36–2,71 — intervalo de parâmetros, não de previsão; não incorpora variabilidade ano-a-ano nem superdispersão).
* Premissas do modelo 8.7 (declarar no TCC): efeito **linear** da idade no log-odds (midpoints das faixas como pontos-etários); linhas agregadas mun×ano×faixa tratadas como **independentes** (superdispersão por município não capturada — ICs otimistas); a tabela previstos × observados da 8.7 é a evidência de adequação do ajuste. A tendência linear da 8.4 é **descritiva** (série com forma em V, R² baixo) — o efeito ano defensável é o do modelo (OR = 0,927).

### O que a tabela PERMITE
* Taxa de alteração ponderada por **faixa etária, ano e município** (razão de somas, com IC de Wilson para bases pequenas).
* **Padrão etário** do resultado alterado (via modelo logístico binomial: OR por década de idade).
* **Tendência temporal** 2020–2025 e projeção ilustrativa de taxa para 2026.
* **Ranking de municípios** com comparação estatisticamente honesta (IC 95%).

### O que a tabela NÃO permite
* **Motivo da alteração (verificado até a bronze em 03/nov/2026):** `bronze.resultados_raw` está no formato largo original do TABNET com colunas apenas de **município + 15 faixas etárias + Total**; os arquivos-fonte são unicamente os 6 CSVs `q5-siscan_colo_alterados_faixa_etaria_*.csv` (2020–2025); e o volume de origem (`siscan_files_bronze/dataset`) contém apenas as 4 famílias de relatórios já ingeridas (Q1/Q2 exames geral, Q3 faixa etária, Q4 adequabilidade, Q5 alterados) — **nenhum relatório com dimensão de diagnóstico**. Não há quebra por ASC-US/LSIL/HSIL/carcinoma em nenhuma camada, bronze incluída. Para decompor o “alterado” seria necessário exportar do SISCAN/TABNET o relatório de resultados por diagnóstico e ingeri-lo no pipeline bronze→silver→gold.
* Distribuição etária **exata** das amostras insatisfatórias/rejeitadas — o SISCAN não cruza adequabilidade × faixa etária, então as exclusões do denominador são alocadas proporcionalmente ao volume de exames de cada faixa (pressuposto metodológico declarável).
* Prever resultado de um **exame individual** (dados agregados — falácia ecológica).
* Ajustar por fatores de confusão individuais (história reprodutiva, HPV, etc.).
* Diagnóstico de causa — alterado ≠ necessariamente lesão (inclui atipias benignas/ASC-US).

> **Conferência de integridade:** o numerador da gold × silver (município/ano) é validado no notebook `05_auditoria_meses_silver` (seção 4).

**Ressalva:** resultados históricos e descritivos. As associações não demonstram causalidade, cobertura individual ou diagnóstico tardio; o agrupamento de alto grau é um proxy operacional dos rótulos TABNET.

## 9. Composição por categoria e município — somente leitura

Distribuição das oito categorias e evolução anual. O agrupamento de alto grau, microinvasora e carcinoma é um proxy de gravidade dos rótulos da extração, não classificação clínica individual.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

df_cat = spark.table("`tcc-analise-siscan-10crs`.gold.fato_categoria_anual")

ORDEM_CATEGORIAS = [
    "ASC-US", "ASC-H", "baixo_grau_HPV-NIC I", "alto_grau_HPV-NIC I",
    "carcinoma-epiderm", "micro-invasa",
    "glandul_naoNeoplasicas", "glandul_nao_afasta_lesao",
]

n_linhas = df_cat.count()
soma = df_cat.agg(F.sum("quantidade_alterados").alias("s")).collect()[0]["s"]
n_mun = df_cat.select("municipio").distinct().count()
n_cat = df_cat.select("categoria").distinct().count()
n_ano = df_cat.select("ano").distinct().count()

print(f"Linhas: {n_linhas} (esperado 528 = 11 mun × 8 categorias × 6 anos)")
print(f"Soma de alterados: {soma} (esperado 4.479 — fechamento regional)")
print(f"Dimensões: {n_mun} municípios × {n_cat} categorias × {n_ano} anos")

assert n_linhas == 528 and soma == 4479 and n_mun == 11 and n_cat == 8 and n_ano == 6
print("Guardas de integridade OK")

In [0]:
qtd = (
    df_cat.groupBy("municipio", "categoria")
    .agg(F.sum("quantidade_alterados").alias("qtd"))
    .toPandas()
)

matriz = (
    qtd.pivot(index="municipio", columns="categoria", values="qtd")
    .reindex(columns=ORDEM_CATEGORIAS)
    .fillna(0)
    .astype(int)
)
matriz["Total"] = matriz.sum(axis=1)

comp_pct = matriz[ORDEM_CATEGORIAS].div(matriz["Total"], axis=0) * 100
comp_pct["% Alto grau"] = matriz[ALTO_GRAU].sum(axis=1).div(matriz["Total"]) * 100
comp_pct = comp_pct.round(2).sort_values("% Alto grau", ascending=False)

matriz_reg = matriz.copy()
matriz_reg.loc["REGIONAL (10ª CRS)"] = matriz.sum()
display(matriz_reg.sort_values("Total", ascending=False))

print("\nComposição (%) por município — ordenada pela proporção de lesões de alto grau:")
display(comp_pct)

In [0]:
hm = comp_pct.loc[comp_pct.index, ORDEM_CATEGORIAS]  # mun já ordenados p/ alto grau

fig, ax = plt.subplots(figsize=(13, 6.5))
im = ax.imshow(hm.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=75)
ax.set_xticks(range(len(hm.columns)), hm.columns, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(hm.index)), hm.index, fontsize=9)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.values[i, j]:.1f}", ha="center", va="center",
                fontsize=8, color="black")
ax.set_title(
    "Composição dos exames alterados por categoria (%) — consolidado 2020–2025\n"
    "(municípios ordenados pela proporção de lesões de alto grau)"
)
fig.colorbar(im, ax=ax, label="% do total de alterados do município")
plt.tight_layout()
plt.show()

In [0]:
evol = (
    df_cat.groupBy("ano", "categoria")
    .agg(F.sum("quantidade_alterados").alias("qtd"))
    .toPandas()
)
evol["total"] = evol.groupby("ano")["qtd"].transform("sum")
evol["pct"] = evol["qtd"] / evol["total"] * 100

resumo_ano = evol.pivot(index="ano", columns="categoria", values="pct")[ORDEM_CATEGORIAS].round(2)
resumo_ano["Total alterados"] = evol.groupby("ano")["total"].first().astype(int)
resumo_ano["% Alto grau"] = (
    evol[evol["categoria"].isin(ALTO_GRAU)].groupby("ano")["pct"].sum().round(2)
)
display(resumo_ano)

fig, ax = plt.subplots(figsize=(10, 5))
for cat, cor in [("ASC-US", "#4c72b0"), ("baixo_grau_HPV-NIC I", "#55a868"),
                 ("alto_grau_HPV-NIC I", "#c44e52")]:
    ax.plot(resumo_ano.index, resumo_ano[cat], marker="o", label=cat, color=cor)
ax.set_title("Evolução anual da composição regional das categorias (%)")
ax.set_xlabel("Ano")
ax.set_ylabel("% dos alterados")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
alto = (
    df_cat.filter(F.col("categoria").isin(ALTO_GRAU))
    .groupBy("municipio")
    .agg(F.sum("quantidade_alterados").alias("n_alto_grau"))
    .toPandas()
    .set_index("municipio")
)
alto["n_total"] = matriz["Total"]
alto["pct"] = (alto["n_alto_grau"] / alto["n_total"] * 100).round(2)
alto = alto.sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
cores = ["#c44e52" if p >= 10 else ("#dd8452" if p >= 6.18 else "#55a868") for p in alto["pct"]]
ax.barh(alto.index, alto["pct"], color=cores)
ax.axvline(6.18, color="black", linestyle="--", linewidth=1, label="Regional 6,18%")
for i, (mun, row) in enumerate(alto.iterrows()):
    ax.text(row["pct"] + 0.15, i, f"{row['pct']:.2f}% ({row['n_alto_grau']}/{row['n_total']})",
            va="center", fontsize=9)
ax.set_title(
    "Lesões de alto grau (NIC I alto + micro-invasora + carcinoma)\n"
    "como % dos exames alterados por município — consolidado 2020–2025"
)
ax.set_xlabel("% dos alterados")
ax.legend(loc="lower right")
ax.set_xlim(0, alto["pct"].max() * 1.3)
plt.tight_layout()
plt.show()

def wilson_p(k: float, n: float, z: float = 1.96) -> tuple[float, float]:
    lo, hi = wilson(k, n, z)
    return round(lo, 2), round(hi, 2)

wil = alto.copy()
wil["IC95_Wilson_pct"] = [wilson_p(k, n) for k, n in zip(wil["n_alto_grau"], wil["n_total"])]
wil["acima_da_regional"] = wil["IC95_Wilson_pct"].apply(lambda ic: ic[0] > 6.18)

Z_BONF = 2.837
wil["IC_Bonferroni_pct"] = [wilson_p(k, n, z=Z_BONF) for k, n in zip(wil["n_alto_grau"], wil["n_total"])]
wil["acima_bonferroni"] = wil["IC_Bonferroni_pct"].apply(lambda ic: ic[0] > 6.18)
display(wil[["n_alto_grau", "n_total", "pct", "IC95_Wilson_pct", "acima_da_regional",
             "IC_Bonferroni_pct", "acima_bonferroni"]]
        .sort_values("pct", ascending=False))

### 9.6 Síntese — composição por categoria × município — resultados históricos

**Achados (consolidado 2020–2025, 4.479 alterados):**
* Composição regional: **ASC-US 68,7%** (3.076), NIC I baixo grau **16,6%** (744), NIC I alto grau **5,1%** (229), ASC-H 4,0% (181), glandulares não neoplásicas 3,7% (167), micro-invasora 0,9% (39), glandulares não afasta lesão 0,8% (34), carcinoma 0,2% (9). Lesões de alto grau somam **6,18%** (277/4.479).
* **Sant'Ana do Livramento destoa: 13,07% de alto grau** (40/306), incluindo **10 das 39 micro-invasoras regionais (25,6%)** — o mesmo município com a **pior cobertura relativa à meta** (31,9% do necessário; auditoria Q3) e a menor taxa global de alterados (1,69%; seção 8.5). Padrão compatível com a hipótese “rastreamento insuficiente → menos achados, porém mais avançados no momento da detecção”.
* **São Gabriel** é o segundo (10,90%; 35/321) e concentra 7 das 39 micro-invasoras. Pelo IC 95% de Wilson (9.5), **apenas Sant'Ana do Livramento [9,75–17,31] e São Gabriel [7,94–14,79] são estatisticamente acima** da regional — nos demais municípios o intervalo contém 6,18%. São 11 ICs a 95% comparados ao regional sem correção de multiplicidade; corrigindo com **Bonferroni (z = 2,837; revisão 09/nov/2026), ambos permanecem acima** — Sant'Ana [8,54–19,50] e São Gabriel [6,89–16,83] — e nenhum outro município cruza o limiar.
* Macambara (11,11%) e Barra do Quarai (4,76%) têm apenas 9 e 21 alterados no período — proporções instáveis, não distinguíveis da regional pelo IC de Wilson (9.5). No extremo oposto, **Alegrete** (2º maior volume, 905 alterados) tem a **menor** proporção de alto grau (4,53%), seguido de Uruguaiana (5,06%) — o maior volume (1.740; 38,9% do total regional).
* Evolução regional: a participação de alto grau cai de **10,32% (2020) para 3,04% (2025)**, enquanto o ASC-US atinge o pico em 2023 (74,45%) — coerente com a retomada do rastreamento pós-COVID detectando mais achados precoces e menos lesões avançadas.

**Limitações:**
* A fonte não cruza categoria × faixa etária × ano — composição por faixa apenas consolidada (seção 5c).
* Divergência ±1–7 por município × ano entre as famílias “categorias” e “alterados” (limitação 5b, notebook 05) — fechamento regional idêntico (4.479).
* Composição municipal reflete também acesso a laboratório/registro (sensibilidade local); a diferença de composição não é, isolada, evidência de diferença epidemiológica.
* 2020 tem menor volume (465) — proporções anuais menos estáveis no início da série.

**Próximo cruzamento sugerido:** proporção de alto grau × cobertura/população-alvo (`fato_indicador_populacao_alvo`) para testar a hipótese “menor cobertura → lesões mais avançadas”.

**Ressalva:** resultados históricos e descritivos. As associações não demonstram causalidade, cobertura individual ou diagnóstico tardio; o agrupamento de alto grau é um proxy operacional dos rótulos TABNET.

## 10. Associação entre categorias e razão anual — somente leitura

Compara o proxy de alto grau entre alterados, razão anual por 100 mulheres (Censo 2022) e proporção na população-alvo. Spearman com permutação e tendência Theil–Sen; associação ecológica não demonstra causalidade.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

CAT = "`tcc-analise-siscan-10crs`"

alto_mun = (
    spark.table(f"{CAT}.gold.fato_categoria_anual")
    .groupBy("codigo_municipio", "municipio")
    .agg(
        F.sum(F.when(F.col("categoria").isin(ALTO_GRAU), F.col("quantidade_alterados"))
             .otherwise(F.lit(0))).alias("n_alto_grau"),
        F.sum("quantidade_alterados").alias("n_alterados"),
    )
    .toPandas()
)
alto_mun["pct_alto_grau"] = alto_mun["n_alto_grau"] / alto_mun["n_alterados"] * 100

cobertura_mun = (
    spark.table(f"{CAT}.gold.fato_indicador_populacao_alvo").alias("i")
    .join(
        spark.table(f"{CAT}.gold.fato_evolucao_cobertura_municipios").alias("e"),
        (F.col("i.codigo_municipio") == F.col("e.codigo_municipio"))
        & (F.col("i.ano") == F.col("e.ano")),
        "inner",
    )
    .groupBy(F.col("i.codigo_municipio"))
    .agg(
        F.avg("e.razao_anual_exames_pct").alias("media_razao_anual"),
        F.avg("i.proporcao_populacao_alvo_pct").alias("media_prop_alvo"),
    )
    .toPandas()
)

cruz_mun = (
    alto_mun.merge(cobertura_mun, on="codigo_municipio")
    .sort_values("pct_alto_grau", ascending=False)
    .reset_index(drop=True)
)

assert len(cruz_mun) == 11
assert cruz_mun["n_alterados"].sum() == 4479 and cruz_mun["n_alto_grau"].sum() == 277
print(f"Guardas OK: {len(cruz_mun)} municípios, {cruz_mun['n_alterados'].sum()} alterados,"
      f" {cruz_mun['n_alto_grau'].sum()} de alto grau")

display(cruz_mun[["municipio", "n_alto_grau", "n_alterados", "pct_alto_grau",
                  "media_razao_anual", "media_prop_alvo"]].round(2))

# Permutação bilateral com semente fixa para a amostra de 11 municípios.
def spearman_com_p(df, a, b, n_perm=20000, seed=42):
    rho = df[[a, b]].corr(method="spearman").iloc[0, 1]
    rx = df[b].rank().to_numpy()
    ry = df[a].rank().to_numpy()
    rng = np.random.default_rng(seed)
    extremos = 0
    for _ in range(n_perm):
        rp = np.corrcoef(rng.permutation(ry), rx)[0, 1]
        if abs(rp) >= abs(rho) - 1e-12:
            extremos += 1
    return rho, (extremos + 1) / (n_perm + 1)

rho_razao, p_razao = spearman_com_p(cruz_mun, "pct_alto_grau", "media_razao_anual")
rho_alvo, p_alvo = spearman_com_p(cruz_mun, "pct_alto_grau", "media_prop_alvo")
print(f"\nSpearman (11 mun) — alto grau × razão anual: ρ = {rho_razao:+.3f} (p permutação ≈ {p_razao:.3f})")
print(f"Spearman (11 mun) — alto grau × % exames na pop-alvo: ρ = {rho_alvo:+.3f} (p permutação ≈ {p_alvo:.3f})")

In [0]:
alto_ano = (
    spark.table(f"{CAT}.gold.fato_categoria_anual")
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(
        F.sum(F.when(F.col("categoria").isin(ALTO_GRAU), F.col("quantidade_alterados"))
             .otherwise(F.lit(0))).alias("n_alto_grau"),
        F.sum("quantidade_alterados").alias("n_alterados"),
    )
    .filter(F.col("n_alterados") > 0)
    .toPandas()
)
alto_ano["pct_alto_grau"] = alto_ano["n_alto_grau"] / alto_ano["n_alterados"] * 100

cobertura_ano = (
    spark.table(f"{CAT}.gold.fato_indicador_populacao_alvo")
    .select("codigo_municipio", "ano", "proporcao_populacao_alvo_pct")
    .join(
        spark.table(f"{CAT}.gold.fato_evolucao_cobertura_municipios")
        .select("codigo_municipio", "ano", "razao_anual_exames_pct"),
        ["codigo_municipio", "ano"],
    )
    .toPandas()
)
cruz_ano = alto_ano.merge(cobertura_ano, on=["codigo_municipio", "ano"])
rho_ano, _ = spearman_com_p(cruz_ano, "pct_alto_grau", "razao_anual_exames_pct")
print(f"Spearman anual ({len(cruz_ano)} pares mun×ano) — ρ = {rho_ano:+.3f} "
      f"(mediana de alterados por par = {cruz_ano['n_alterados'].median():.0f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
ax.scatter(cruz_mun["media_razao_anual"], cruz_mun["pct_alto_grau"], s=70, color="#c44e52", zorder=3)
for _, r in cruz_mun.iterrows():
    ax.annotate(r["municipio"], (r["media_razao_anual"], r["pct_alto_grau"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
# Theil–Sen nas unidades dos eixos, não em ranks.
xv = cruz_mun["media_razao_anual"].to_numpy()
yv = cruz_mun["pct_alto_grau"].to_numpy()
i_, j_ = np.triu_indices(len(xv), k=1)
dx = xv[j_] - xv[i_]
validos = dx != 0
exigir(bool(validos.any()), "Theil–Sen: nenhuma diferença válida no eixo x")
ts_slope = np.median((yv[j_] - yv[i_])[validos] / dx[validos])
ts_b = np.median(yv - ts_slope * xv)
xs = np.linspace(xv.min(), xv.max(), 50)
ax.plot(xs, ts_b + ts_slope * xs, "--", color="#4c72b0", alpha=0.8,
        label=f"Spearman ρ = {rho_razao:+.2f} · Theil–Sen {ts_slope:+.2f} p.p./1% razão")
ax.set_xlabel("Razão anual de exames 25–64 por 100 mulheres (média 2020–2025, %)")
ax.set_ylabel("% de alto grau entre os alterados")
ax.set_title("Consolidado por município (2020–2025)")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(cruz_ano["razao_anual_exames_pct"], cruz_ano["pct_alto_grau"], s=28, alpha=0.6, color="#55a868")
ax.set_xlabel("Razão anual de exames (%)")
ax.set_ylabel("% de alto grau")
ax.set_title(f"Por município × ano ({len(cruz_ano)} pares) — ρ = {rho_ano:+.2f}")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 10.3 Síntese — alto grau × cobertura — resultados históricos

**Achados (11 municípios, 2020–2025):**
* **ρ = −0,53** entre a razão anual de exames (média por município) e o % de alto grau entre os alterados (p ≈ 0,10 por permutação — limítrofe, não significativa ao nível de 5% com n = 11): municípios que examinam menos por 100 mulheres da população-alvo tendem a detectar lesões **mais avançadas** quando detectam — direção coerente com a hipótese da seção 9.6.
* Os extremos puxam a correlação: **Sant'Ana** (razão média 10,5%; 13,07% de alto grau) e **São Gabriel** (11,4%; 10,90%) versus **Manoel Viana** (24,5%; 5,96%), **Alegrete** (19,4%; 4,53%) e **Itaqui** (17,8%; 6,04%). A reta tracejada do gráfico consolidado é agora um ajuste **Theil–Sen nas unidades brutas** (revisão 09/nov/2026): **−0,34 p.p. de alto grau por 1 ponto percentual de razão anual** (intercepto +11,8).
* **ρ = +0,67** com o % de exames na população-alvo (25–64) — significativo (p ≈ 0,03 por permutação): efeito **composicional** — lesões de alto grau concentram-se nas idades-alvo, enquanto o sobrerastreamento <25 predomina ASC-US; municípios que concentram os exames em 25–64 (Itaqui: 96,4%) acham proporcionalmente mais alto grau.
* No nível anual (66 pares mun × ano) o sinal enfraquece (ρ = −0,16; p ≈ 0,20 por permutação): contagens anuais pequenas (mediana ≈ 40 alterados por par) dominam o ruído — o agregado consolidado por município é a leitura adequada.

**Limitações:**
* Correlação ecológica (11 unidades agregadas) — não estabelece causalidade nem descreve indivíduos (falácia ecológica).
* n = 11 e municípios com 9–21 alterados no período (Macambara, Barra do Quarai) tornam o p-valor frágil; Sant'Ana é forte driver do resultado.
* Razão anual usa denominador IBGE 2022 fixo (limitação 3c); o % de alto grau reflete também acesso a laboratório e qualidade de registro.
* A direção negativa é consistente com rastreamento insuficiente, mas explicações alternativas (perfil etário populacional, triagem laboratorial diferenciada) não podem ser descartadas com estes dados agregados.

**Ressalva:** resultados históricos e descritivos. As associações não demonstram causalidade, cobertura individual ou diagnóstico tardio; o agrupamento de alto grau é um proxy operacional dos rótulos TABNET.

## 11. Categorias por faixa etária — somente leitura

Composição consolidada 2020–2025, sem dimensão anual. Intervalos de Wilson são exploratórios; bases pequenas aumentam a incerteza.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

CAT = "`tcc-analise-siscan-10crs`"
ORDEM_FAIXA = [
    "10 a 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
    "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos",
    "60 a 64 anos", "65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos",
]

cat_faixa = spark.table(f"{CAT}.gold.fato_categoria_faixa_etaria")

n_linhas = cat_faixa.count()
assert n_linhas == 11 * 8 * 15, f"Esperado 1.320 linhas; obtido {n_linhas}"
soma = cat_faixa.agg(F.sum("quantidade_alterados")).collect()[0][0]
assert soma == 4479, f"Esperado 4.479 alterados; obtido {soma}"
div = (
    cat_faixa.groupBy("categoria").agg(F.sum("quantidade_alterados").alias("n5c"))
    .join(
        spark.table(f"{CAT}.gold.fato_categoria_anual")
        .groupBy("categoria").agg(F.sum("quantidade_alterados").alias("n5b")),
        "categoria",
    )
    .filter(F.col("n5c") != F.col("n5b"))
    .count()
)
assert div == 0, f"{div} categorias divergem entre 5c e 5b"
print(f"Guardas OK: {n_linhas} linhas, {soma} alterados, grade 11×8×15, 5c ≡ 5b por categoria")

pdf = (
    cat_faixa.groupBy("faixa_etaria", "categoria")
    .agg(F.sum("quantidade_alterados").alias("qtd"))
    .toPandas()
)
tot_faixa = pdf.groupby("faixa_etaria")["qtd"].transform("sum")
pdf["pct"] = pdf["qtd"] / tot_faixa * 100

pivot_qtd = pdf.pivot(index="faixa_etaria", columns="categoria", values="qtd").reindex(ORDEM_FAIXA).fillna(0).astype(int)
pivot_pct = pdf.pivot(index="faixa_etaria", columns="categoria", values="pct").reindex(ORDEM_FAIXA).fillna(0)

print("Alterados por categoria em cada faixa etária (regional, 2020–2025):")
display(pivot_qtd)
print("\nComposição (%) — soma 100% por faixa:")
display(pivot_pct.round(1))

In [0]:
import numpy as np
import matplotlib.pyplot as plt

g = (
    cat_faixa
    .withColumn("alto", F.when(F.col("categoria").isin(ALTO_GRAU), F.col("quantidade_alterados")).otherwise(F.lit(0)))
    .groupBy("faixa_etaria")
    .agg(F.sum("alto").alias("n_alto"), F.sum("quantidade_alterados").alias("n"))
    .toPandas()
    .set_index("faixa_etaria")
    .reindex(ORDEM_FAIXA)
)

g["pct_alto"] = g["n_alto"] / g["n"] * 100
ci = np.array([wilson(k, n) for k, n in zip(g["n_alto"], g["n"])])
g["ic_lo"], g["ic_hi"] = np.maximum(0, ci[:, 0]), np.minimum(100, ci[:, 1])
print(g[["n_alto", "n", "pct_alto", "ic_lo", "ic_hi"]].round(2).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(g))
ax.bar(x, g["pct_alto"], color="#c44e52", zorder=3)
ax.errorbar(x, g["pct_alto"],
            yerr=[g["pct_alto"] - g["ic_lo"], g["ic_hi"] - g["pct_alto"]],
            fmt="none", ecolor="#4c72b0", capsize=4, zorder=4)
ax.axhline(6.18, ls="--", color="gray", label="Regional (todas as idades): 6,18%")
ax.set_xticks(x)
ax.set_xticklabels([f.replace(" anos", "") for f in g.index], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("% de alto grau entre os alterados")
ax.set_title("Lesões de alto grau por faixa etária — regional, 2020–2025 (IC 95% Wilson)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [0]:
def grupo_etario(faixa: str) -> str:
    i = ORDEM_FAIXA.index(faixa)
    if i < 3:
        return "<25 anos"
    if i >= 11:
        return "65+ anos"
    return "25–64 anos"

gm = (
    cat_faixa.groupBy("municipio", "faixa_etaria")
    .agg(F.sum("quantidade_alterados").alias("qtd"))
    .toPandas()
)
gm["grupo"] = gm["faixa_etaria"].map(grupo_etario)

qt = gm.pivot_table(index="municipio", columns="grupo", values="qtd", aggfunc="sum").fillna(0)
qt = qt.reindex(columns=["<25 anos", "25–64 anos", "65+ anos"])
qt.loc["REGIONAL"] = qt.sum()

pct = qt.div(qt.sum(axis=1), axis=0) * 100
final = qt.astype(int).join(pct.round(2), lsuffix=" (n)", rsuffix=" (%)")
print("Perfil etário dos alterados por município (2020–2025) — ordenado pelo % <25 anos:")
display(final.sort_values("<25 anos (%)", ascending=False))

### 11.4 Síntese — composição por faixa etária — resultados históricos

**Achados (regional, 2020–2025):**
* **Gradiente etário nítido do alto grau:** 0% aos 10–14 anos, 0,48% [IC 0,08–2,65] aos 15–19 e 1,65% [0,76–3,56] aos 20–24; salta para 4,92% [3,33–7,21] aos 25–29 e culmina em **8,94% [6,70–11,83] (30–34) e 9,38% [7,19–12,16] (35–39)**. Por grupos da diretriz: **1,2% (<25) · 6,9% (25–64) · 7,8% (65+)** vs 6,18% regional — lesão avançada antes dos 25 é praticamente inexistente (7 de 4.479 no período).
* **Idades avançadas:** o % volta a subir (75–79: 13,79% [5,50–30,56]; acima de 79: 28,57% [8,22–64,11]) — porém n=29 e n=7, ICs largos. Padrão compatível com sub-rastreamento (detecção mais tardia), eco do efeito de Sant'Ana das seções 9 e 10.
* **Categorias dominantes por idade:** ASC-US domina TODAS as faixas (57–80%); o NIC I baixo cai de ~29% (15–24 anos) para 6–8% (55+), enquanto o NIC I alto compõe mais aos 30–39 (7,3–8,3% da faixa).
* **Perfil municipal espelha a clientela examinada:** Uruguaiana tem o maior % de alterados <25 (16,9%) — coerente com seu maior sobrerastreamento de exames (14,6%) e menor % alto grau (5,06%); Itaqui concentra 97,1% dos alterados na pop-alvo — coerente com o ρ = +0,67 da seção 10 e seu 6,04%. Barra do Quarai lidera em % <25 (23,8%) mas com apenas 21 alterados — leitura instável.

**Limitações:**
* Consolidado 2020–2025 **sem dimensão anual** — a fonte TABNET por categoria não cruza faixa × ano (ver introdução da seção 10); a evolução temporal da composição está na seção 9.
* Extremos etários com n minúsculo (10–14: 9; acima de 79: 7) — os ICs Wilson registram a incerteza; não interpretar pontos isolados.
* Divergência conhecida ±1–7 por município vs `fato_exames_alterados` (downloads TABNET distintos: por faixa vs por categoria); fechamento regional idêntico (4.479).
* Composições por razão de somas; valores 0 em células da grade zero-fill significam “sem caso observado” (não “impossível”).

**Ressalva:** resultados históricos e descritivos. As associações não demonstram causalidade, cobertura individual ou diagnóstico tardio; o agrupamento de alto grau é um proxy operacional dos rótulos TABNET.